# Notebook 14: Clustering — Unsupervised Learning
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Distinguish supervised from unsupervised learning
- Apply K-Means and choose K with Elbow and Silhouette methods
- Build and read a dendrogram for hierarchical clustering
- Apply DBSCAN to find clusters of arbitrary shape
- Evaluate clustering quality with appropriate metrics

## 1. What Is Clustering?

In **unsupervised learning** there are no labels — we want to discover structure in the data.

**Clustering** groups similar data points together. Applications:
- Customer segmentation
- Document grouping
- Anomaly detection
- Gene expression analysis
- Image compression

**Key challenge**: we have no ground truth to validate against directly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

## 2. K-Means

**Algorithm**:
1. Choose K initial centroids (K-Means++ initialisation)
2. Assign each point to its nearest centroid
3. Recompute each centroid as the mean of its assigned points
4. Repeat 2–3 until convergence

**Objective**: minimise within-cluster sum of squares (WCSS / inertia):
$$\text{WCSS} = \sum_{k=1}^K \sum_{\mathbf{x} \in C_k} \|\mathbf{x} - \boldsymbol{\mu}_k\|^2$$

In [ ]:
X_blob, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
km = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
labels = km.fit_predict(X_blob)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(X_blob[:,0], X_blob[:,1], c=y_true, cmap='Set1', s=30, alpha=0.7)
axes[0].set_title('True Clusters')

axes[1].scatter(X_blob[:,0], X_blob[:,1], c=labels, cmap='Set1', s=30, alpha=0.7)
axes[1].scatter(km.cluster_centers_[:,0], km.cluster_centers_[:,1],
                marker='*', s=400, c='black', zorder=5, label='Centroids')
axes[1].set_title(f'K-Means  ARI={adjusted_rand_score(y_true, labels):.3f}')
axes[1].legend()
plt.tight_layout(); plt.show()

## 3. Choosing K — Elbow and Silhouette

**Elbow**: plot WCSS vs K — look for where the curve bends.

**Silhouette score** $s$ for a point:
$$s = \frac{b - a}{\max(a, b)}$$
where $a$ = average distance to same-cluster points, $b$ = average distance to nearest other-cluster points.
$s \in [-1, 1]$: higher = better clustering.

In [ ]:
inertias, silhouettes = [], []
for k in range(2, 11):
    km_k = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    lbl = km_k.fit_predict(X_blob)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(X_blob, lbl))

best_k = np.argmax(silhouettes) + 2
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(range(2,11), inertias, marker='o')
axes[0].set_xlabel('K'); axes[0].set_ylabel('WCSS'); axes[0].set_title('Elbow Method')
axes[1].plot(range(2,11), silhouettes, marker='s', color='darkorange')
axes[1].axvline(best_k, color='red', linestyle='--', label=f'Best K={best_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].legend()
plt.tight_layout(); plt.show()
print(f'Best K by silhouette: {best_k}')

## 4. Hierarchical Clustering

**Agglomerative** (bottom-up):
1. Each point is its own cluster
2. Merge the two closest clusters
3. Repeat until one cluster remains

The **dendrogram** visualises the merge history. Cut at any height to get a flat clustering.

In [ ]:
X_hier, _ = make_blobs(n_samples=40, centers=4, cluster_std=0.6, random_state=1)
Z = linkage(X_hier, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dendrogram(Z, ax=axes[0], leaf_rotation=90, color_threshold=4)
axes[0].axhline(4, color='red', linestyle='--', label='Cut → 4 clusters')
axes[0].set_title('Dendrogram (Ward linkage)')
axes[0].set_xlabel('Sample'); axes[0].set_ylabel('Distance')
axes[0].legend()

ag = AgglomerativeClustering(n_clusters=4, linkage='ward')
axes[1].scatter(X_hier[:,0], X_hier[:,1], c=ag.fit_predict(X_hier), cmap='Set1', s=60)
axes[1].set_title('Agglomerative Clustering (K=4)')
plt.tight_layout(); plt.show()

## 5. DBSCAN — Density-Based Clustering

DBSCAN finds clusters of **arbitrary shape** based on local density:
- **Core point**: ≥ `min_samples` neighbours within radius `eps`
- **Border point**: within `eps` of a core point
- **Noise point**: neither → labelled −1 (outlier)

**Advantages**: no need to specify K; finds non-convex clusters; identifies outliers automatically.

In [ ]:
X_moon, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_circ, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for row, (X_d, name) in enumerate([(X_moon,'Moons'),(X_circ,'Circles')]):
    for col, (alg, lbl) in enumerate([
        (KMeans(n_clusters=2, random_state=42),           'K-Means'),
        (AgglomerativeClustering(n_clusters=2),            'Agglomerative'),
        (DBSCAN(eps=0.2, min_samples=5),                   'DBSCAN'),
    ]):
        ax = axes[row][col]
        preds = alg.fit_predict(X_d)
        ax.scatter(X_d[:,0], X_d[:,1], c=preds, cmap='Set1', s=15, alpha=0.8)
        noise = (preds==-1).sum() if hasattr(alg,'labels_') and -1 in preds else 0
        ax.set_title(f'{name}: {lbl}' + (f' (noise={noise})' if noise else ''))
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

## Exercises

1. Apply K-Means to the Iris dataset (without labels). Does the Silhouette score suggest K=3 (the true number of species)? Measure ARI.
2. Try `linkage='single'`, `'complete'`, `'average'`, and `'ward'`. Which produces the most compact clusters?
3. For the moons dataset, vary `eps = [0.05, 0.1, 0.2, 0.5]`. What happens when eps is too small? Too large?
4. Apply PCA (2 components) to the breast cancer dataset, cluster with K-Means (K=2), and visualise. Do clusters align with benign/malignant?

## Reflection Questions
- K-Means assumes spherical, equal-sized clusters. Give two real-world examples where this assumption fails badly.
- Why must you scale features before K-Means but not before a tree-based method?
- If you cluster customers and one cluster is labelled "high-value," what ethical issues arise from automated decisions based on this?

---
**Next →** `15_deep_learning_foundations.ipynb`